## book_p31, transparent-background SVG loss curves for Figure 3.15

Book render of `P31`. Same model / image / layers / colors / sweep, but the frames are exported as
**transparent-background SVGs** instead of black-background PNGs, following the `book_p30` house style
(`svg.fonttype = 'none'`, one shared crop box, `summary.json` beside the curves).

Figure 3.15 (Right), *High Dimensional Space is Perilous*, is the grid of small screens each showing one
parameter's loss curve, the point being that the model moves through **1.45M** of these at once. So this
notebook renders many curves as individual tiles for layout, rather than one composed figure.

Each of the five p27 layers contributes its top-|dL/dw| parameters; each is swept over ±2.5 with every other
weight at its trained value, and rendered as `−ln P(correct)` with that curve's own min/max (+5%) as y-limits.

Two variants per curve, so the layout can pick:

- **`chrome`**, spines, tick marks, tick labels and grid in `CHILL_BROWN` (the `P31` look)
- **`nolabels`**, the same box, grid and tick marks, but no numbers
- **`bare`**, the curve path alone on transparency

Both variants share **one** crop box across all curves, so every tile drops into a grid at identical scale
and the axes land in the same place.

```
book_p31_curves/layer_k/chrome/grad_NN.svg k = 1..5 in the p27 ordering (1 = fc, magenta)
book_p31_curves/layer_k/nolabels/grad_NN.svg NN = gradient rank 00..
book_p31_curves/layer_k/bare/grad_NN.svg
book_p31_curves/layer_k/curves.npy (n x 3 x N): [theta, P, loss] per rank
```

Nothing else is written, the render settings live in the config cells above, not in a sidecar file.

In [ ]:
BLUE="#2ca3dd"
YELLOW="#ffd35a"
RED='#ec2027'
CHILL_BROWN="#948979"

In [ ]:
import json, gc, io, time
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.transforms import Bbox
from torchvision.models.resnet import ResNet, BasicBlock
import torchvision.datasets as dsets, torchvision.transforms as T
from PIL import Image
from IPython.display import display

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['svg.fonttype'] = 'none'

RUNS_CANDIDATES = [
    Path("/home/stephen/Stephencwelch Dropbox/welch_labs/resnet/hackin/aug_17_run"),
    Path("~/Stephencwelch Dropbox/welch_labs/resnet/hackin/aug_17_run").expanduser(),
]
GFX_CANDIDATES = [
    Path("/home/stephen/Stephencwelch Dropbox/welch_labs/ai_book_vol_2/3_resnets/graphics"),
    Path("~/Stephencwelch Dropbox/welch_labs/ai_book_vol_2/3_resnets/graphics").expanduser(),
]
BOOK_DATA_CANDIDATES = [
    Path("../../../ai_book_vol_2/chapters/03-resnets/data"),
    Path("~/Documents/ai_book_vol_2/chapters/03-resnets/data").expanduser(),
]
IMAGENET = Path("/home/stephen/imagenet")

RUNS = next((p for p in RUNS_CANDIDATES if p.exists()), None)

def runs_dir(name):
    '''Directory containing <name>/final.pt. Off Stephen's boxes this pulls from the HF mirror,
    fetching only the one model asked for (the whole runs/ tree is ~770MB). The dataset is
    private: export HF_TOKEN, or `huggingface-cli login`, before running.'''
    if RUNS is not None:
        return RUNS
    from huggingface_hub import snapshot_download
    return Path(snapshot_download(repo_id="WelchLabs/ai_book_vol_2_data", repo_type="dataset",
                                  allow_patterns=f"chapters/03-resnets/runs/{name}/*")) / "chapters/03-resnets/runs"

GFX = next((p for p in GFX_CANDIDATES if p.exists()), None)

NB_DIR = next((p for p in [Path.cwd(), Path("~/Documents/videos/_2026/resnets").expanduser()]
               if (p / 'book_p31.ipynb').exists()), Path.cwd())
OUT = (GFX / 'p31_curves_svg') if GFX else (NB_DIR / 'book_p31_curves')
BOOK_DATA = next((p for p in BOOK_DATA_CANDIDATES if p.exists()), None)

device = "cuda" if torch.cuda.is_available() else \
         "mps" if torch.backends.mps.is_available() else "cpu"

MODEL_DEPTHS = {"plain8": 8, "plain14": 14, "plain20": 20, "plain26": 26,
                "plain34": 34, "plain56": 56, "plain74": 74, "resnet74": 74}
print(torch.__version__, device)
print('runs:', RUNS or '(HF mirror, fetched per model)')
print('out :', OUT)

### Model + data (same as P25_37c / P31)

In [ ]:
class PlainBasicBlock(BasicBlock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.downsample = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

LAYER_CFG = {8: [1,1,1,1], 14: [2,1,1,2], 20: [2,2,3,2], 26: [3,3,3,3],
             34: [4,4,4,4], 56: [3,4,17,3], 74: [3,4,26,3]}

def make_net(depth_target, use_skip, num_classes=1000):
    block = BasicBlock if use_skip else PlainBasicBlock
    model = ResNet(block, LAYER_CFG[depth_target], num_classes=num_classes)
    if depth_target == 8:
        model.layer4 = nn.Identity()
        model.fc = nn.Linear(256, num_classes)
    return model

def load_model(name, step=None):
    d = runs_dir(name) / name
    path = d / "final.pt" if step is None else d / f"ckpt_step{step:06d}.pt"
    model = make_net(MODEL_DEPTHS[name], use_skip=name.startswith("resnet"))
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

def weighted_layers(model):
    '''(name, module) for convs + fc in forward order, excluding 1x1 shortcuts.'''
    return [(n, m) for n, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear)) and "downsample" not in n]

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])

if IMAGENET.exists():
    val_ds = dsets.ImageFolder(IMAGENET / "ILSVRC/Data/CLS-LOC/val", _tf)

    def load_image(idx):
        '''ImageNet val index -> (x, y).'''
        x, y = val_ds[idx]
        return x.unsqueeze(0).to(device), torch.tensor([y], device=device)
else:
    CLASS_NAMES = [line.strip().split(" ", 1)[1].split(",")[0]
                   for line in open(BOOK_DATA / "LOC_synset_mapping.txt")]

    def load_image(idx, jpg="screwdriver.jpg", cls="screwdriver"):
        '''Fallback: the book repo's jpg, labelled via LOC_synset_mapping. `idx` is ignored.'''
        x = _tf(Image.open(BOOK_DATA / jpg).convert("RGB")).unsqueeze(0).to(device)
        return x, torch.tensor([CLASS_NAMES.index(cls)], device=device)

crit = nn.CrossEntropyLoss()

@torch.no_grad()
def correct_ans_conf(model, x, y):
    return F.softmax(model(x), dim=1)[0, y.item()].item()

def layer_grad(model, layer, x, y):
    '''dL/dw for one layer at the current weights.'''
    model.zero_grad(set_to_none=True)
    crit(model(x), y).backward()
    g = layer.weight.grad.detach().flatten().clone()
    model.zero_grad(set_to_none=True)
    return g

@torch.no_grad()
def sweep_weight(model, layer, flat_idx, x, y, span, n):
    '''Set one weight to each value in linspace(-span, span, n), record P(correct), restore. Returns (vals, res, w0).'''
    w = layer.weight.view(-1)
    w0 = w[flat_idx].item()
    vals = np.linspace(-span, span, n)
    res = np.empty(n)
    for i, v in enumerate(vals):
        w[flat_idx] = v
        res[i] = correct_ans_conf(model, x, y)
    w[flat_idx] = w0
    return vals, res, w0

def theta_latex(coord, layer_num):
    inner = r',\,'.join(str(int(c)) for c in coord)
    return r'$\theta_{(' + inner + r')}^{(' + str(layer_num) + r')}$'

## Config

In [ ]:
MODEL_NAME = 'plain8'
IMG_IDX    = 39209
SPAN       = 2.5
N_POINTS   = 128
N_CURVES   = 96
RANK_STRIDE = 1
PAD_FRAC   = 0.05
P_FLOOR    = 1e-12

LAYERS = [
    dict(k=1, li=-1, color='m'),
    dict(k=2, li=-3, color='#eb8423'),
    dict(k=3, li=3,  color='#419c52'),
    dict(k=4, li=2,  color='#ed5e78'),
    dict(k=5, li=0,  color='#d73b2f'),
]

for L, part in zip(LAYERS, np.array_split(np.arange(N_CURVES), len(LAYERS))):
    L['n'] = len(part)
print('curves per layer:', {L['k']: L['n'] for L in LAYERS}, '= ', sum(L['n'] for L in LAYERS))

In [ ]:
model = load_model(MODEL_NAME)
x, y  = load_image(IMG_IDX)
y_true = y.item()
base_conf = correct_ans_conf(model, x, y)
layers = weighted_layers(model)
print(f"P(true) at trained weights = {base_conf:.4f}   (loss {-np.log(base_conf):.4f})")

### Rank + sweep: top-|grad| parameters per layer

In [ ]:
for L in LAYERS:
    name, layer = layers[L['li']]
    g = layer_grad(model, layer, x, y)
    _, idxs = g.abs().topk(L['n'] * RANK_STRIDE)
    grad_ranks = list(range(0, L['n'] * RANK_STRIDE, RANK_STRIDE))
    idxs = idxs[::RANK_STRIDE]
    L.update(name=name, layer=layer, layer_num=L['li'] % len(layers) + 1, entries=[])
    for rank, flat_idx in enumerate(tqdm(idxs.tolist(), desc=f"layer {L['k']} {name}", leave=False)):
        vals, res, w0 = sweep_weight(model, layer, flat_idx, x, y, SPAN, N_POINTS)
        loss = -np.log(np.clip(res, P_FLOOR, None))
        lo, hi = loss.min(), loss.max(); pad = PAD_FRAC * (hi - lo)
        coord = tuple(int(c) for c in np.unravel_index(flat_idx, layer.weight.shape))
        L['entries'].append(dict(rank=rank, grad_rank=grad_ranks[rank], flat_idx=flat_idx, coord=coord, w0=w0, grad=g[flat_idx].item(),
                                 vals=vals, res=res, loss=loss, loss_ylim=(lo - pad, hi + pad),
                                 theta_label=theta_latex(coord, L['layer_num'])))
    assert abs(correct_ans_conf(model, x, y) - base_conf) < 1e-6, "weights not restored!"

for L in LAYERS:
    e0, eN = L['entries'][0], L['entries'][-1]
    print(f"layer {L['k']}  {L['name']:<16} L{L['layer_num']}  |grad| rank0 = {abs(e0['grad']):.4f}  rank{eN['grad_rank']} = {abs(eN['grad']):.4f}   "
          f"loss ranges: rank0 {e0['loss'].min():.2f}..{e0['loss'].max():.2f}, rank{eN['grad_rank']} {eN['loss'].min():.2f}..{eN['loss'].max():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6))
for ax, L in zip(axes, LAYERS):
    for e in L['entries']:
        ax.plot(e['vals'], e['loss'], c=L['color'], lw=1, alpha=0.5)
    ax.set_title(f"layer {L['k']}: {L['name']} (L{L['layer_num']})", fontsize=10)
    ax.set_xlim(-SPAN, SPAN); ax.grid(alpha=0.3)
axes[0].set_ylabel(r'$-\ln P$');

### Renderer + shared crop

Everything the layout might want to retune lives in the block below: axis color, spines, tick marks,
tick labels, grid, curve weight. `chrome` is the `P31` look; `bare` is the curve alone.

In [ ]:
FIGSIZE     = (6, 6)
CURVE_LW    = 5
SHOW_W0_DOT = False
DOT_SIZE    = 125

AXIS_COLOR = CHILL_BROWN
TICK_FS    = 12
TICK_LEN   = 3.5
GRID_ALPHA = 0.3
GRID_LW    = 0.8

VARIANTS = {
    'chrome':   dict(spines=True,  ticks=True,  tick_labels=True,  grid=True),
    'nolabels': dict(spines=True,  ticks=True,  tick_labels=False, grid=True),
    'bare':     dict(spines=False, ticks=False, tick_labels=False, grid=False),
}

def style_ax(ax, v):
    ax.set_facecolor('none')
    ax.spines[:].set_visible(v['spines'])
    if v['spines']:
        ax.spines[:].set_color(AXIS_COLOR)
    ax.tick_params(which='both', colors=AXIS_COLOR, labelsize=TICK_FS,
                   length=TICK_LEN if v['ticks'] else 0,
                   left=v['ticks'], bottom=v['ticks'],
                   labelleft=v['tick_labels'], labelbottom=v['tick_labels'])
    if v['grid']:
        ax.grid(True, color=AXIS_COLOR, alpha=GRID_ALPHA, linewidth=GRID_LW)

def fig_loss(e, color, v):
    '''Transparent-background loss curve styled by one VARIANTS spec.'''
    fig = Figure(figsize=FIGSIZE)
    ax = fig.add_subplot(111)
    ax.plot(e['vals'], e['loss'], lw=CURVE_LW, c=color)
    if SHOW_W0_DOT:
        ax.scatter(e['w0'], -np.log(max(base_conf, P_FLOOR)), s=DOT_SIZE, c=color, zorder=10)
    ax.set_xlim(-SPAN, SPAN); ax.set_ylim(*e['loss_ylim'])
    style_ax(ax, v)
    return fig

def save_svg(fig, path, bbox):
    fig.savefig(path, format='svg', bbox_inches=bbox, transparent=True)
    fig.clear()

def tight(fig):
    FigureCanvasAgg(fig)
    return fig.get_tightbbox(fig.canvas.get_renderer())

boxes = []
for L in LAYERS:
    for e in L['entries']:
        for v in VARIANTS.values():
            f = fig_loss(e, L['color'], v); boxes.append(tight(f)); f.clear()
BBOX = Bbox.union(boxes).padded(0.1)
print('crop (inches):', np.round(BBOX.bounds, 3))

In [ ]:
def _checker(size, n=16):
    a = np.indices((size[1] // n + 1, size[0] // n + 1)).sum(0) % 2
    im = np.kron(a, np.ones((n, n))) [:size[1], :size[0]]
    return Image.fromarray((200 + 30 * im).astype('uint8')).convert('RGB')

def preview(width=200):
    '''First / middle / last rank per layer, chrome and bare side by side.'''
    rows = []
    for L in LAYERS:
        es = [L['entries'][0], L['entries'][len(L['entries']) // 2], L['entries'][-1]]
        row = []
        for e in es:
            for v in VARIANTS.values():
                buf = io.BytesIO()
                f = fig_loss(e, L['color'], v)
                f.savefig(buf, format='png', dpi=100, bbox_inches=BBOX, transparent=True); f.clear()
                buf.seek(0)
                im = Image.open(buf).convert('RGBA')
                im = im.resize((width, int(width * im.height / im.width)))
                bg = _checker(im.size); bg.paste(im, (0, 0), im)
                row.append(bg)
        rows.append(row)
    W, H = rows[0][0].size
    sheet = Image.new('RGB', (W * len(rows[0]), H * len(rows)), 'white')
    for r, row in enumerate(rows):
        for c, im in enumerate(row): sheet.paste(im, (c * W, r * H))
    display(sheet)
preview()

## Render

In [ ]:
for L in LAYERS:
    d = OUT / f"layer_{L['k']}"
    for v in VARIANTS: (d / v).mkdir(parents=True, exist_ok=True)
    for e in tqdm(L['entries'], desc=f"layer {L['k']}", leave=False):
        for name, v in VARIANTS.items():
            save_svg(fig_loss(e, L['color'], v), d / name / f"grad_{e['rank']:02d}.svg", BBOX)
    gc.collect()
    np.save(d / 'curves.npy', np.stack([np.stack([e['vals'], e['res'], e['loss']]) for e in L['entries']]))

files = sorted(OUT.rglob('grad_*.svg'))
print(len(files), 'svgs written to', OUT)
assert len(files) == sum(L['n'] for L in LAYERS) * len(VARIANTS)